# Phase 1 Legal LM — resumable training with periodic Kaggle Dataset checkpoint backups

This notebook initializes a decoder-only GPT-style language model **from random weights** and trains for up to **3,000 optimizer steps** (set `TARGET_STEPS = 2_000` for a 2,000-step run). It is designed for Kaggle GPU sessions that may stop unexpectedly.

## Important reliability notes
- **A Kaggle session cannot reliably tell you how many minutes remain before Kaggle stops it.** This notebook therefore uses periodic saves/uploads rather than relying on a last-minute warning.
- It writes a local, atomic checkpoint every `LOCAL_SAVE_EVERY` steps, and attempts to publish a checkpoint snapshot as a **new version of a Kaggle Dataset** every `UPLOAD_EVERY` steps.
- Dataset uploads take time and can fail (network/session termination/API quota/credentials). Keep local checkpoints and download the final checkpoint too. No notebook can guarantee a final upload if the runtime is killed abruptly.
- To resume after a disconnect, attach the checkpoint dataset as an input to the new Kaggle notebook, then rerun the notebook. It automatically selects the highest-step checkpoint it finds.
- You must configure the Kaggle Dataset slug and Kaggle API credentials in the setup cell. Do not paste credentials into a shared notebook.
- The model is initialized from scratch. This is a generic GPT-2-style decoder setup matching the `GPT2LMHeadModel` checkpoint conventions from the earlier inference notebook; adjust dimensions to match your intended Phase 1 architecture. It does not restore a prior model unless you intentionally enable resume.


In [ ]:

import os
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
os.environ["KAGGLE_USERNAME"] = secrets.get_secret("KAGGLE_USERNAME")
os.environ["KAGGLE_KEY"] = secrets.get_secret("KAGGLE_KEY")

print("Kaggle credentials loaded.")


In [ ]:
# 1. Configuration — edit this cell before running

import os, json, time, random, shutil, hashlib, subprocess, sys
from pathlib import Path
from datetime import datetime

# Training target
TARGET_STEPS = 3000             # change to 2000 if preferred
LOCAL_SAVE_EVERY = 50            # local atomic checkpoint frequency
UPLOAD_EVERY = 250               # attempt Kaggle Dataset version upload every N steps
UPLOAD_ON_FINISH = True
UPLOAD_ON_EXCEPTION = True
MAX_UPLOAD_RETRIES = 2
UPLOAD_RETRY_SLEEP_SECONDS = 20

# Model: set these to your Phase 1 decoder architecture.
VOCAB_SIZE = 32000
CONTEXT_LENGTH = 1024            # reduce if GPU memory is tight; must fit your token batches
HIDDEN_SIZE = 640
NUM_LAYERS = 8
NUM_HEADS = 8
DROPOUT = 0.1

# Training
BATCH_SIZE = 1                   # conservative default for T4; increase only after checking VRAM
GRAD_ACCUM_STEPS = 8
LEARNING_RATE = 3e-4
WEIGHT_DECAY = 0.1
WARMUP_STEPS = 100
GRAD_CLIP_NORM = 1.0
SEED = 42
USE_AMP = True                   # uses BF16 if supported, otherwise FP16
NUM_WORKERS = 0

# Paths
WORK_DIR = Path("/kaggle/working/phase1_legal_lm")
CHECKPOINT_DIR = WORK_DIR / "checkpoints"
LOG_PATH = WORK_DIR / "train_log.jsonl"
WORK_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

# Kaggle Dataset that will store snapshots, e.g. "your-kaggle-username/cs781-phase1-checkpoints"
# Create the dataset once (or use an existing writable dataset) and set this to owner/dataset-slug.
KAGGLE_DATASET_SLUG = "oipranesh/cs781-phase1-checkpoints"

# If using Kaggle Secrets, set KAGGLE_USERNAME and KAGGLE_KEY in notebook environment/secrets.
# Otherwise configure ~/.kaggle/kaggle.json yourself (never commit it to a public notebook).
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

print("Target steps:", TARGET_STEPS)
print("Work directory:", WORK_DIR)
print("Checkpoint dataset:", KAGGLE_DATASET_SLUG)


In [ ]:
# 2. Install / import dependencies
%pip -q install -U transformers tokenizers kaggle

import torch
import torch.nn.functional as F
from torch.utils.data import IterableDataset, DataLoader
from transformers import GPT2Config, GPT2LMHeadModel
from tokenizers import Tokenizer

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("BF16 supported:", torch.cuda.is_bf16_supported())
else:
    raise RuntimeError("No CUDA GPU found. In Kaggle, enable a GPU accelerator in Notebook Settings.")


In [ ]:
# 3. Seed and locate tokenizer + legal training text
random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

INPUT_ROOT = Path("/kaggle/input")

def find_first_tokenizer():
    preferred = [
        INPUT_ROOT / "datasets/oipranesh/cs781-phase4-training-prep/tokenizer/legal_bpe_32k.json",
        INPUT_ROOT / "datasets/oipranesh/cs781-phase4-final-recovery/tokenizer/legal_bpe_32k.json",
    ]
    for p in preferred:
        if p.exists():
            return p
    matches = list(INPUT_ROOT.rglob("legal_bpe_32k.json"))
    if matches:
        return matches[0]
    matches = list(INPUT_ROOT.rglob("tokenizer.json"))
    if matches:
        return matches[0]
    raise FileNotFoundError("Tokenizer JSON not found under /kaggle/input. Attach the dataset containing legal_bpe_32k.json.")

TOKENIZER_PATH = find_first_tokenizer()
tokenizer = Tokenizer.from_file(str(TOKENIZER_PATH))
tok_vocab = tokenizer.get_vocab_size()
print("Tokenizer:", TOKENIZER_PATH, "| vocab size:", tok_vocab)
if tok_vocab != VOCAB_SIZE:
    print(f"WARNING: configured VOCAB_SIZE={VOCAB_SIZE}, tokenizer vocab={tok_vocab}. Setting VOCAB_SIZE to tokenizer vocab.")
    VOCAB_SIZE = tok_vocab

# Extract strings from JSON/JSONL training files, excluding obvious test/validation/submission files.
EXCLUDED_HINTS = ("test", "valid", "validation", "prediction", "submission", "sample_submission")
INCLUDED_HINTS = ("train", "corpus", "legal", "acts", "court", "tribunal", "judgment", "judgement")

def extract_texts(obj):
    out = []
    if isinstance(obj, str):
        if len(obj.strip()) >= 80:
            out.append(obj.strip())
    elif isinstance(obj, dict):
        # Prioritize conventional text fields; recursively inspect if absent.
        for k in ("text", "content", "document", "body", "judgment", "judgement"):
            v = obj.get(k)
            if isinstance(v, str) and len(v.strip()) >= 80:
                out.append(v.strip())
        if not out:
            for v in obj.values():
                out.extend(extract_texts(v))
    elif isinstance(obj, list):
        for v in obj:
            out.extend(extract_texts(v))
    return out

def read_json_file(path):
    try:
        raw = path.read_text(encoding="utf-8", errors="ignore")
        if path.suffix.lower() == ".jsonl":
            texts = []
            for line in raw.splitlines():
                if line.strip():
                    try: texts.extend(extract_texts(json.loads(line)))
                    except Exception: pass
            return texts
        return extract_texts(json.loads(raw))
    except Exception as e:
        print("Skipping unreadable file:", path, "|", type(e).__name__)
        return []

candidate_files = []
for p in INPUT_ROOT.rglob("*"):
    if not p.is_file() or p.suffix.lower() not in (".json", ".jsonl"):
        continue
    low = str(p).lower()
    if any(h in low for h in EXCLUDED_HINTS):
        continue
    if any(h in low for h in INCLUDED_HINTS):
        candidate_files.append(p)

print("Candidate training files:", len(candidate_files))
for p in candidate_files[:20]:
    print(" -", p)

if not candidate_files:
    raise FileNotFoundError(
        "No likely legal training JSON/JSONL files found. Attach the Phase 1 training dataset or edit candidate_files selection."
    )

# Load and tokenize once. This is simple/reliable but can consume RAM for a very large corpus.
# For a multi-billion-token corpus, use pre-tokenized shards or replace this with a streaming shard dataset.
all_ids = []
seen_docs = 0
for p in candidate_files:
    texts = read_json_file(p)
    if texts:
        print(f"{p.name}: {len(texts)} text records")
    for txt in texts:
        try:
            ids = tokenizer.encode(txt).ids
            if len(ids) >= 16:
                all_ids.extend(ids)
                all_ids.append(tokenizer.token_to_id("[EOS]") or tokenizer.token_to_id("<|endoftext|>") or 0)
                seen_docs += 1
        except Exception:
            continue
    # optional RAM guard; remove this cap if you have enough RAM / use shard streaming
    if len(all_ids) >= 30_000_000:
        print("Reached 30M-token RAM guard; stopping corpus ingestion.")
        break

if len(all_ids) < CONTEXT_LENGTH + 2:
    raise RuntimeError(f"Not enough tokenized training data: {len(all_ids)} tokens.")

print(f"Loaded {seen_docs:,} documents and {len(all_ids):,} tokens.")
print("WARNING: This simple loader concatenates the selected legal training text in RAM. For very large corpora, use pre-tokenized shards.")


In [ ]:
# 4. Build fixed-length next-token batches
class PackedTokenDataset(IterableDataset):
    def __init__(self, ids, seq_len, seed=42):
        self.ids = ids
        self.seq_len = seq_len
        self.seed = seed
    def __iter__(self):
        rng = random.Random(self.seed + int(time.time()) // 60)
        n = len(self.ids) - self.seq_len - 1
        while True:
            start = rng.randint(0, n)
            chunk = self.ids[start:start + self.seq_len + 1]
            x = torch.tensor(chunk[:-1], dtype=torch.long)
            y = torch.tensor(chunk[1:], dtype=torch.long)
            yield x, y

train_loader = DataLoader(
    PackedTokenDataset(all_ids, CONTEXT_LENGTH, SEED),
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
)
batch_iter = iter(train_loader)


In [ ]:
# 5. Initialize model FROM SCRATCH
config = GPT2Config(
    vocab_size=VOCAB_SIZE,
    n_positions=CONTEXT_LENGTH,
    n_ctx=CONTEXT_LENGTH,
    n_embd=HIDDEN_SIZE,
    n_layer=NUM_LAYERS,
    n_head=NUM_HEADS,
    resid_pdrop=DROPOUT,
    embd_pdrop=DROPOUT,
    attn_pdrop=DROPOUT,
    bos_token_id=tokenizer.token_to_id("[BOS]"),
    eos_token_id=tokenizer.token_to_id("[EOS]"),
)
model = GPT2LMHeadModel(config)  # random initialization; no pretrained weights
device = torch.device("cuda")
model.to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f"Parameters: {n_params:,} ({n_params/1e6:.1f}M)")
if n_params < 50_000_000 or n_params > 300_000_000:
    print("WARNING: Phase 1 project requirement says 50M–300M parameters; adjust HIDDEN_SIZE/NUM_LAYERS if needed.")

optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scaler = torch.amp.GradScaler("cuda", enabled=(USE_AMP and not torch.cuda.is_bf16_supported()))
use_bf16 = bool(USE_AMP and torch.cuda.is_bf16_supported())
print("AMP:", USE_AMP, "| BF16:", use_bf16, "| GradScaler:", scaler.is_enabled())


In [ ]:
# 6. Atomic checkpointing + Kaggle Dataset version upload
# Kaggle API requires an existing dataset you own and credentials with write permission.
# First-time setup: create the dataset manually in Kaggle, add dataset-metadata.json, and set KAGGLE_DATASET_SLUG.

import signal, traceback

def atomic_torch_save(obj, destination: Path):
    destination.parent.mkdir(parents=True, exist_ok=True)
    tmp = destination.with_suffix(destination.suffix + ".tmp")
    torch.save(obj, tmp)
    os.replace(tmp, destination)

def save_checkpoint(step, loss_value=None, keep_snapshot=False):
    payload = {
        "step": int(step),
        "model_config": {
            "vocab_size": VOCAB_SIZE,
            "context_length": CONTEXT_LENGTH,
            "hidden_size": HIDDEN_SIZE,
            "num_layers": NUM_LAYERS,
            "num_heads": NUM_HEADS,
            "dropout": DROPOUT,
        },
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scaler_state_dict": scaler.state_dict() if scaler.is_enabled() else None,
        "loss": float(loss_value) if loss_value is not None else None,
        "random_state": random.getstate(),
        "torch_rng_state": torch.get_rng_state(),
        "cuda_rng_state_all": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None,
        "saved_at": datetime.utcnow().isoformat() + "Z",
    }
    latest = CHECKPOINT_DIR / "checkpoint_latest.pt"
    atomic_torch_save(payload, latest)
    # Keep step-specific snapshots only at upload milestones to avoid filling disk.
    if keep_snapshot:
        atomic_torch_save(payload, CHECKPOINT_DIR / f"checkpoint_step_{step:06d}.pt")
    return latest

def kaggle_credentials_ready():
    token_file = Path.home() / ".kaggle" / "kaggle.json"
    return (bool(os.environ.get("KAGGLE_USERNAME")) and bool(os.environ.get("KAGGLE_KEY"))) or token_file.exists()

def upload_checkpoint(step):
    if "YOUR_KAGGLE_USERNAME" in KAGGLE_DATASET_SLUG:
        print("UPLOAD SKIPPED: set KAGGLE_DATASET_SLUG to owner/dataset-slug.")
        return False
    if not kaggle_credentials_ready():
        print("UPLOAD SKIPPED: configure KAGGLE_USERNAME/KAGGLE_KEY or ~/.kaggle/kaggle.json.")
        return False

    # Upload only the latest checkpoint + metadata, not every historical duplicate.
    upload_dir = WORK_DIR / "dataset_upload"
    if upload_dir.exists():
        shutil.rmtree(upload_dir)
    upload_dir.mkdir(parents=True)
    shutil.copy2(CHECKPOINT_DIR / "checkpoint_latest.pt", upload_dir / "checkpoint_latest.pt")
    meta = {
        "title": "CS781 Phase 1 Legal LM Checkpoints",
        "id": KAGGLE_DATASET_SLUG,
        "licenses": [{"name": "CC0-1.0"}],
    }
    (upload_dir / "dataset-metadata.json").write_text(json.dumps(meta, indent=2))
    (upload_dir / "README.md").write_text(
        f"# CS781 Phase 1 checkpoint backup\n\nLatest uploaded optimizer step: {step}\n"
        "This dataset contains a resumable training checkpoint created by the owner.\n"
    )
    for attempt in range(1, MAX_UPLOAD_RETRIES + 1):
        try:
            # --dir-mode tar avoids one-file-per-file upload overhead and is broadly supported.
            cmd = ["kaggle", "datasets", "version", "-p", str(upload_dir),
                   "-m", f"Automatic resumable checkpoint at step {step}", "--dir-mode", "tar"]
            print("Uploading checkpoint snapshot (attempt", attempt, ") ...")
            result = subprocess.run(cmd, capture_output=True, text=True, timeout=900)
            print(result.stdout[-3000:])
            if result.returncode == 0:
                print("Checkpoint dataset version upload succeeded.")
                return True
            print("Upload failed:", result.stderr[-3000:])
        except Exception as e:
            print("Upload exception:", repr(e))
        if attempt < MAX_UPLOAD_RETRIES:
            time.sleep(UPLOAD_RETRY_SLEEP_SECONDS)
    print("WARNING: upload failed; local checkpoint is still saved.")
    return False

print("Checkpoint functions ready.")


In [ ]:
# 7. Resume from a checkpoint if one is available (optional)
# Set RESUME_FROM_CHECKPOINT=True to resume. Otherwise, training starts from random weights.
RESUME_FROM_CHECKPOINT = True

def find_resume_checkpoint():
    candidates = []
    # Existing working directory
    candidates.extend(CHECKPOINT_DIR.glob("checkpoint_*.pt"))
    # Attached Kaggle input datasets
    for p in INPUT_ROOT.rglob("checkpoint_latest.pt"):
        candidates.append(p)
    for p in INPUT_ROOT.rglob("checkpoint_step_*.pt"):
        candidates.append(p)
    candidates = [p for p in candidates if p.is_file() and not p.name.endswith(".tmp")]
    if not candidates:
        return None
    def step_from_path(p):
        try:
            ck = torch.load(p, map_location="cpu", weights_only=False)
            return int(ck.get("step", 0))
        except Exception:
            return -1
    return max(candidates, key=step_from_path)

start_step = 0
if RESUME_FROM_CHECKPOINT:
    resume_path = find_resume_checkpoint()
    if resume_path:
        print("Resume candidate:", resume_path)
        ckpt = torch.load(resume_path, map_location="cpu", weights_only=False)
        # Verify architectural compatibility before loading weights.
        expected = (VOCAB_SIZE, CONTEXT_LENGTH, HIDDEN_SIZE, NUM_LAYERS, NUM_HEADS)
        mc = ckpt.get("model_config", {})
        found = (mc.get("vocab_size"), mc.get("context_length"), mc.get("hidden_size"), mc.get("num_layers"), mc.get("num_heads"))
        if found != expected:
            raise ValueError(f"Checkpoint architecture {found} does not match current config {expected}. Set the same config as the saved run.")
        model.load_state_dict(ckpt["model_state_dict"])
        if "optimizer_state_dict" in ckpt:
            optimizer.load_state_dict(ckpt["optimizer_state_dict"])
        if scaler.is_enabled() and ckpt.get("scaler_state_dict"):
            scaler.load_state_dict(ckpt["scaler_state_dict"])
        start_step = int(ckpt.get("step", 0))
        print("Resumed at step", start_step)
    else:
        print("No checkpoint found; starting from random initialization.")
else:
    print("Resume disabled; starting from random initialization.")


In [ ]:
# 8. Training loop with periodic local saves and dataset uploads
# Note: periodic upload is the best available protection; Kaggle does not expose a reliable
# imminent-session-timeout callback. A disconnect between upload intervals can lose recent steps.

def append_log(record):
    with LOG_PATH.open("a", encoding="utf-8") as f:
        f.write(json.dumps(record) + "\n")

def emergency_save_and_upload(signum=None, frame=None):
    print("\nReceived termination signal; saving latest checkpoint...")
    try:
        save_checkpoint(current_step, last_loss)
        upload_checkpoint(current_step)
    except Exception:
        traceback.print_exc()
    if signum is not None:
        raise KeyboardInterrupt(f"Signal {signum}")

# SIGTERM may be delivered for some shutdown paths, but Kaggle may kill the process without warning.
signal.signal(signal.SIGTERM, emergency_save_and_upload)

current_step = start_step
last_loss = None
model.train()
optimizer.zero_grad(set_to_none=True)
train_started = time.time()

try:
    for step in range(start_step + 1, TARGET_STEPS + 1):
        current_step = step
        batch_losses = []
        for micro in range(GRAD_ACCUM_STEPS):
            x, y = next(batch_iter)
            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)
            with torch.autocast(
                device_type="cuda",
                dtype=torch.bfloat16 if use_bf16 else torch.float16,
                enabled=USE_AMP,
            ):
                outputs = model(input_ids=x, labels=y)
                loss = outputs.loss / GRAD_ACCUM_STEPS
            if scaler.is_enabled():
                scaler.scale(loss).backward()
            else:
                loss.backward()
            batch_losses.append(float(loss.detach().item()) * GRAD_ACCUM_STEPS)

        if scaler.is_enabled():
            scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
        if scaler.is_enabled():
            scaler.step(optimizer)
            scaler.update()
        else:
            optimizer.step()
        optimizer.zero_grad(set_to_none=True)

        # Simple warmup + cosine decay across this target run.
        if step <= WARMUP_STEPS:
            lr = LEARNING_RATE * step / max(1, WARMUP_STEPS)
        else:
            progress = min(1.0, (step - WARMUP_STEPS) / max(1, TARGET_STEPS - WARMUP_STEPS))
            lr = LEARNING_RATE * 0.5 * (1.0 + __import__("math").cos(__import__("math").pi * progress))
        for group in optimizer.param_groups:
            group["lr"] = lr

        last_loss = sum(batch_losses) / len(batch_losses)
        if step == 1 or step % 10 == 0:
            elapsed = time.time() - train_started
            print(f"step {step:5d}/{TARGET_STEPS} | loss {last_loss:.4f} | lr {lr:.2e} | elapsed {elapsed/60:.1f} min")
            append_log({"step": step, "loss": last_loss, "lr": lr, "elapsed_seconds": elapsed, "time": datetime.utcnow().isoformat() + "Z"})

        if step % LOCAL_SAVE_EVERY == 0 or step == TARGET_STEPS:
            save_checkpoint(step, last_loss, keep_snapshot=(step % UPLOAD_EVERY == 0 or step == TARGET_STEPS))
            print("Local checkpoint saved at step", step)

        if step % UPLOAD_EVERY == 0:
            # Save before network operations, so the local copy is safe even if upload fails.
            upload_checkpoint(step)

    print("Training reached target step", TARGET_STEPS)
    if UPLOAD_ON_FINISH:
        save_checkpoint(current_step, last_loss)
        upload_checkpoint(current_step)

except KeyboardInterrupt:
    print("Training interrupted. Saving current progress.")
    save_checkpoint(current_step, last_loss)
    if UPLOAD_ON_EXCEPTION:
        upload_checkpoint(current_step)
    raise
except Exception:
    print("Training error. Saving current progress before raising.")
    traceback.print_exc()
    try:
        save_checkpoint(current_step, last_loss)
        if UPLOAD_ON_EXCEPTION:
            upload_checkpoint(current_step)
    finally:
        raise

print("Finished. Latest checkpoint:", CHECKPOINT_DIR / "checkpoint_latest.pt")


## Before starting: one-time Kaggle Dataset setup

1. In Kaggle, create a **private dataset** you own for checkpoint backups, e.g. `cs781-phase1-checkpoints`. Its slug must match `KAGGLE_DATASET_SLUG` in Cell 1.
2. Add Kaggle API credentials using Kaggle Secrets (`KAGGLE_USERNAME`, `KAGGLE_KEY`) or place `kaggle.json` at `~/.kaggle/kaggle.json` with permissions `600`. Do not publish credentials.
3. Attach the legal training corpus and tokenizer dataset as notebook inputs. Make sure the candidate-file discovery cell prints the intended training files, and excludes test data.
4. Run all cells. On a fresh run, the model is randomly initialized. On a restarted run with the backup dataset attached, it should resume from the checkpoint with the largest `step` value. Ensure the saved model dimensions match the configuration.
5. If upload fails, download `/kaggle/working/phase1_legal_lm/checkpoints/checkpoint_latest.pt` or save notebook output. Kaggle Dataset version uploads are not a substitute for local backups.

### Important implementation caveats
- The corpus loader is intentionally generic. Inspect `candidate_files` and text counts before training; edit it if it picks an unintended file or misses the actual training corpus format.
- It loads tokens into RAM and caps ingestion at 30M tokens. This is a practical starter, not a high-throughput multi-billion-token pipeline.
- `CONTEXT_LENGTH=1024`, `BATCH_SIZE=1`, and gradient accumulation are conservative. Training a 50M–300M model for 3,000 steps on a T4 may be slow; monitor step time and GPU memory.
- Kaggle Dataset uploads create versions and may take several minutes. Avoid uploading every few steps. `UPLOAD_EVERY=250` is a tradeoff; lower it to 100 if you prioritize recovery over speed.
- The notebook saves model and optimizer states so a restart can resume training, not merely load model weights.
